<a href="https://colab.research.google.com/github/aansheeagrwal/python-training/blob/main/DCGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers

In [ ]:
# Hyperparameter ==

Buf_Size = 60000
Bat_Size = 255
Img_Hi = 28
Img_Wi = 28
Channels = 1
Epochs = 30
Lat_Dim = 100

In [ ]:
# load and Preprocess Dataset
def load_dataset():
    (x_train,y_train),(x_test,y_test) = tf.keras.datasets.fashion_mnist.load_data()
    x_train = x_train.reshape((-1,Img_Hi,Img_Wi,Channels)).astype("float32")
    x_train = (x_train-127.5)/127.5
    return x_train

In [ ]:
x_train = load_dataset()

train_dataset = tf.data.Dataset\
.from_tensor_slices(x_train)\
.shuffle(Buf_Size)\
.batch(Bat_Size)\
.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Build Generators --
def Build_generatos():
    model = tf.keras.Sequential(name="Generator")
    model.add(layers.Dense(7*7*255,use_bias=False,input_shape=(Lat_Dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Reshape((7,7,255)))

    # UpSample 14X14
    model.add(layers.Conv2DTranspose(128,kernel_size=5,strides=2,padding="same",use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # 28X28
    model.add(layers.Conv2DTranspose(128,kernel_size=5,strides=2,padding="same",use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # Final layers ==
    model.add(layers.Conv2DTranspose(Channels,kernel_size=5,strides=1,padding="same",use_bias=False,activation="tanh"))
    return model

In [ ]:
generator = Build_generatos()
generator.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12495)          │     1,249,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 12495)          │        49,980 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 12495)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 7, 7, 255)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose                │ (None, 14, 14, 128)    │       816,000 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_1              │ (None, 28, 28, 128)    │       409,600 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_2              │ (None, 28, 28, 1)      │         3,200 │
│ (Conv2DTranspose)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,529,304 (9.65 MB)

 Trainable params: 2,503,802 (9.55 MB)

 Non-trainable params: 25,502 (99.62 KB)

In [ ]:
# Build Discriminators ==
def Build_discriminators():
    model = tf.keras.Sequential(name="Discriminator")
    model.add(layers.Conv2D(64,kernel_size=5,strides=2,
                            padding="same",use_bias=False,
                            input_shape=[Img_Hi,Img_Wi,Channels]))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))
    model.add(layers.Conv2D(128,kernel_size=5,strides=2,
                            padding="same",use_bias=False))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))
    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    return model

In [ ]:
discriminator = Build_discriminators()
discriminator.summary()

Model: "Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │       204,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         6,273 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 212,673 (830.75 KB)

 Trainable params: 212,673 (830.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Losses and Optimizers ==
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def loss_discriminators(real,fake):
    real_loss = cross_entropy(tf.ones_like(real),real)
    fake_loss = cross_entropy(tf.zeros_like(fake),fake)

    return real_loss + fake_loss

def loss_generators(fake):
    return cross_entropy(tf.ones_like(fake),fake)

In [ ]:
# Optimizers ==
generator_optimizers = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizers = tf.keras.optimizers.Adam(1e-4)

In [ ]:
#